# W05 · 라이브 2D LJ NVE ⇄ NVT (덱 5장과 같은 계산 · 원본 03주차 라이브)
환산단위 ε = σ = m = k_B = 1 · Δt 0.005 τ · r_c 2.5σ · 주기경계 · Berendsen τ_T 0.5 τ

In [ ]:
import numpy as np, matplotlib.pyplot as plt
def lj2d(N=4, L=5.0, T0=0.6, nvt=False, tau=0.5, steps=3000, dt=0.005, rc=2.5, seed=7):
    rng = np.random.default_rng(seed); dof = 2*N - 2; rc2 = rc*rc; s6c = 1/rc2**3; ecut = 4*(s6c*s6c - s6c)
    nc = int(np.ceil(np.sqrt(N))); sp = min(L/nc, 1.25); off = (L - sp*(nc-1))/2
    pos = np.array([[off + sp*(i % nc), off + sp*(i // nc)] for i in range(N)]) + (rng.random((N, 2)) - 0.5)*0.12
    vel = rng.random((N, 2)) - 0.5; vel -= vel.mean(0); vel *= np.sqrt(T0*dof/(2*0.5*(vel**2).sum()))
    def forces(p):
        d = p[:, None, :] - p[None, :, :]; d -= L*np.round(d/L); r2 = (d**2).sum(-1); np.fill_diagonal(r2, np.inf)
        m = (r2 < rc2) & (r2 > 0.04); r2m = np.where(m, r2, np.inf); inv = 1/r2m; s6 = inv**3; fr = np.where(m, 24*(2*s6*s6 - s6)*inv, 0)
        return (fr[:, :, None]*d).sum(1), 0.5*np.where(m, 4*(s6*s6 - s6) - ecut, 0).sum(), 0.5*(fr*r2m).sum()
    f, pe, vir = forces(pos); out = []
    for s in range(steps):
        vel += 0.5*dt*f; pos += dt*vel; pos %= L; f, pe, vir = forces(pos); vel += 0.5*dt*f
        K = 0.5*(vel**2).sum(); T = 2*K/dof
        if nvt: lam = np.clip(np.sqrt(1 + dt/tau*(T0/max(T, 1e-6) - 1)), 0.9, 1.1); vel *= lam; K = 0.5*(vel**2).sum(); T = 2*K/dof
        out.append((s*dt, T, K + pe, (K + 0.5*vir)/(L*L)))
    return np.array(out)
nve, nvt = lj2d(nvt=False), lj2d(nvt=True)
fig, ax = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
for a, k, lab in zip(ax, (1, 2, 3), ("T", "E = K + U", "P")):
    a.plot(nve[:, 0], nve[:, k], label="NVE"); a.plot(nvt[:, 0], nvt[:, k], label="NVT (Berendsen 0.5)"); a.set_ylabel(lab)
ax[0].axhline(0.6, color="gray", ls="--"); ax[0].legend(); ax[2].set_xlabel("t (tau)"); fig.tight_layout(); plt.show()
print("NVE E std", nve[:, 2].std().round(4), "· NVT E drift", (nvt[-1, 2] - nvt[0, 2]).round(3), "· NVT T mean", nvt[len(nvt)//2:, 1].mean().round(3))